# Bank Term Deposit Propensity Model
### Business-Oriented Pipeline: EDA → Feature Engineering → Customer Profiling → Modelling

**Objective:** Identify high-propensity customers, optimal contact channel, contact frequency,  
and build a calibrated propensity score for campaign targeting.

**Business Questions Answered:**
1. Which customers should be targeted? (customer profiles)
2. Which contact mode is most effective?
3. How many contacts maximise conversions before diminishing returns?
4. Does previous campaign outcome affect future success?
5. When should campaigns run?
6. Which customer segments yield the highest ROI?


## I. Setup & Imports

In [ ]:
# ── Core ──────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import statsmodels.api as sm
from scipy import stats

# ── Sklearn – Preprocessing & Pipeline ───────────────────────────────────────
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder, RobustScaler, PowerTransformer, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# ── Sklearn – Models ──────────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# ── Sklearn – Evaluation ──────────────────────────────────────────────────────
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
    f1_score
)
from sklearn.inspection import permutation_importance
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

# ── Imbalanced-learn ──────────────────────────────────────────────────────────
try:
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.over_sampling import SMOTE
    IMBLEARN = True
except ImportError:
    IMBLEARN = False
    print("imblearn not found – SMOTE disabled. Install with: pip install imbalanced-learn")

# ── Saving ────────────────────────────────────────────────────────────────────
import joblib

# ── Global style ──────────────────────────────────────────────────────────────
PALETTE   = {'NO': '#E74C3C', 'YES': '#27AE60'}
SNS_PAL   = 'Set2'
FIG_DPI   = 120
sns.set_theme(style='whitegrid', palette=SNS_PAL)
plt.rcParams.update({'figure.dpi': FIG_DPI, 'axes.titlesize': 13,
                     'axes.labelsize': 11, 'xtick.labelsize': 9, 'ytick.labelsize': 9})

print("✅  All libraries loaded successfully.")


## II. Data Loading & Initial Overview

In [ ]:
data = pd.read_csv("bank-full.csv", sep=';')

# Normalise strings: strip whitespace + upper-case
for col in data.select_dtypes('object'):
    data[col] = data[col].str.strip().str.upper()

print(f"Shape: {data.shape[0]:,} rows × {data.shape[1]} columns")
data.head()


In [ ]:
print("─── dtypes & nulls ─────────────────────────────────────")
print(data.info())

print("\n─── descriptive stats ──────────────────────────────────")
data.describe().T.style.background_gradient(cmap='Blues')


In [ ]:
print("─── Null values ─────────────────────────────────────────")
print(data.isna().sum())

print("\n─── Duplicate rows ──────────────────────────────────────")
print(data.duplicated().sum())

print("\n─── 'unknown' counts per column ─────────────────────────")
unk = (data == 'UNKNOWN').sum()
print(unk[unk > 0].to_string())


### Key Observations
1. **No null values** – dataset is complete.
2. **No duplicate rows.**
3. `unknown` values in `job`, `education`, `contact`, `poutcome` carry real business meaning (see EDA section).
4. `duration` causes **target leakage** (call duration is only known *after* a decision is made) → dropped for modelling.
5. `poutcome = UNKNOWN` is valid when a client was never contacted before.


## III. Feature Engineering & Preprocessing

### 3.1 Age Bands
Segment customers into life-stage buckets that align with marketing personas.


In [ ]:
data['age_band'] = pd.cut(
    data['age'],
    bins=[17, 29, 44, 59, 100],
    labels=['YOUNG-ADULT', 'MIDDLE-AGED', 'LATE-MIDDLE-AGED', 'SENIOR'],
    include_lowest=True
).astype('object')

print(data['age_band'].value_counts(normalize=True).mul(100).round(1).to_string())


### 3.2 Balance Tiers

In [ ]:
print("Balance percentile distribution:")
print(data['balance'].describe(percentiles=[.01,.05,.25,.5,.75,.90,.95,.99]))


In [ ]:
data['tier'] = pd.cut(
    data['balance'],
    bins=[-99999, 0, 500, 1500, 6000, 9999999],
    labels=['NEGATIVE', 'LOW', 'MIDDLE', 'HIGH', 'PREMIUM']
).astype('object')

print(data['tier'].value_counts(normalize=True).mul(100).round(1).to_string())


### 3.3 Previous Contact Flag & Sanity Check

In [ ]:
# Sanity-check: previous=0 must imply pdays=-1
data.loc[(data['previous'] == 0) & (data['pdays'] != -1), 'pdays'] = -1
data.loc[(data['previous'] > 0) & (data['pdays'] < 0),  'previous'] = 0
data.loc[(data['previous'] == 0) & (data['pdays'] == 0), 'pdays'] = -1

data['previously_contacted'] = np.where(data['pdays'] == -1, 'NO', 'YES')
data['previous_success']     = np.where(data['poutcome'] == 'SUCCESS', 1, 0)

print("previously_contacted distribution:")
print(data['previously_contacted'].value_counts())


### 3.4 Campaign Contact Buckets

In [ ]:
data['campaign_bucket'] = pd.cut(
    data['campaign'],
    bins=[0, 1, 3, 6, 999],
    labels=['1 contact', '2–3 contacts', '4–6 contacts', '7+ contacts']
).astype('object')

print(data['campaign_bucket'].value_counts().to_string())


## IV. Exploratory Data Analysis

### Fig 0 – Target Distribution (Class Imbalance)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Pie chart
counts = data['y'].value_counts()
axes[0].pie(
    counts, labels=counts.index, autopct='%.1f%%',
    colors=[PALETTE['NO'], PALETTE['YES']],
    startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
axes[0].set_title('Overall Subscription Split')

# Bar with annotation
bars = axes[1].bar(counts.index, counts.values,
                   color=[PALETTE['NO'], PALETTE['YES']], edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
                 f'{val:,}', ha='center', fontsize=10, fontweight='bold')
axes[1].set_title('Subscription Volume')
axes[1].set_ylabel('Number of Customers')
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.suptitle('Fig 0 – Class Imbalance: Only ~11.7% Subscribe', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig0_class_imbalance.png', bbox_inches='tight')
plt.show()
print("⚠️  Dataset is highly imbalanced. Use class_weight='balanced' or SMOTE during modelling.")


### Fig 1 – Normality Diagnostics (Q-Q Plots)
Guides which numerical features need transformation before linear models.


In [ ]:
num_vars = ['age', 'balance', 'campaign', 'previous']

fig, axes = plt.subplots(len(num_vars), 3, figsize=(14, 4 * len(num_vars)))

for i, var in enumerate(num_vars):
    col = data[var]

    # Histogram + KDE
    axes[i, 0].hist(col, bins=40, color='steelblue', edgecolor='white', alpha=0.8, density=True)
    xmin, xmax = axes[i, 0].get_xlim()
    x_vals = np.linspace(xmin, xmax, 200)
    axes[i, 0].plot(x_vals, stats.norm.pdf(x_vals, col.mean(), col.std()),
                    'r--', lw=1.5, label='Normal fit')
    axes[i, 0].set_title(f'{var} – Histogram')
    axes[i, 0].legend(fontsize=8)

    # Box plot
    axes[i, 1].boxplot(col, vert=True, patch_artist=True,
                       boxprops=dict(facecolor='steelblue', alpha=0.6))
    axes[i, 1].set_title(f'{var} – Boxplot')
    axes[i, 1].set_ylabel(var)

    # Q-Q plot
    sm.qqplot(col, line='45', ax=axes[i, 2], alpha=0.4, markersize=2)
    axes[i, 2].set_title(f'{var} – Q-Q Plot')

    # Shapiro-Wilk on a sample (max 5000)
    sample = col.sample(min(5000, len(col)), random_state=42)
    stat, p_val = stats.shapiro(sample)
    skew_val = col.skew()
    print(f"{var:12s}  skew={skew_val:+.2f}  Shapiro-W p={'<0.001' if p_val < 0.001 else f'{p_val:.3f}'}")

plt.suptitle('Fig 1 – Normality Diagnostics for Numeric Features', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig1_normality_diagnostics.png', bbox_inches='tight')
plt.show()
print("\n→ balance is heavily right-skewed → use PowerTransformer (Yeo-Johnson).")
print("→ campaign & previous are count variables → RobustScaler or leave as-is.")


### Fig 2 – Correlation Heatmap
Identify multicollinearity risks before modelling.


In [ ]:
num_for_corr = ['age', 'balance', 'day', 'campaign', 'pdays', 'previous']
corr = data[num_for_corr].corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title('Fig 2 – Correlation Matrix (Numeric Features)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig2_correlation_heatmap.png', bbox_inches='tight')
plt.show()
print("→ pdays & previous show moderate positive correlation (as expected).")
print("→ No severe multicollinearity issues for tree-based models.")


---
## V. Stakeholder Questions

### Q1 – Who Should We Target? (Customer Profiles)


#### Fig 3.1 – Subscription Rate by Age Band

In [ ]:
age_stats = (
    data.groupby('age_band', observed=True)['y']
    .agg(total='count', subscribers=lambda x: (x == 'YES').sum())
    .assign(rate=lambda d: d['subscribers'] / d['total'] * 100)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Volume bars
colors = sns.color_palette(SNS_PAL, len(age_stats))
bars = axes[0].bar(age_stats['age_band'], age_stats['total'], color=colors, edgecolor='white')
axes[0].set_title('Contact Volume by Age Band')
axes[0].set_ylabel('Number of Contacts')
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar, val in zip(bars, age_stats['total']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'{val:,}', ha='center', fontsize=8)

# Conversion rate
bar2 = axes[1].bar(age_stats['age_band'], age_stats['rate'], color=colors, edgecolor='white')
axes[1].set_title('Subscription Rate by Age Band')
axes[1].set_ylabel('Subscription Rate (%)')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
for bar, val in zip(bar2, age_stats['rate']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{val:.1f}%', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Fig 3.1 – Age Band Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig3_1_age_band.png', bbox_inches='tight')
plt.show()

print("\nAge Band Summary:")
print(age_stats.to_string(index=False))


#### Fig 3.2 – Subscription Rate by Job

In [ ]:
job_stats = (
    data.groupby('job')['y']
    .agg(total='count', subscribers=lambda x: (x == 'YES').sum())
    .assign(rate=lambda d: d['subscribers'] / d['total'] * 100)
    .sort_values('rate', ascending=True)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Subscription rate (horizontal bar)
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(job_stats)))
bars = axes[0].barh(job_stats['job'], job_stats['rate'], color=colors)
axes[0].set_xlabel('Subscription Rate (%)')
axes[0].set_title('Subscription Rate by Job')
axes[0].xaxis.set_major_formatter(mtick.PercentFormatter())
for bar, val in zip(bars, job_stats['rate']):
    axes[0].text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=8, fontweight='bold')

# Volume
job_vol = job_stats.sort_values('total', ascending=True)
axes[1].barh(job_vol['job'], job_vol['total'],
             color=sns.color_palette(SNS_PAL, len(job_vol)))
axes[1].set_xlabel('Number of Contacts')
axes[1].set_title('Contact Volume by Job')
axes[1].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.suptitle('Fig 3.2 – Job Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig3_2_job_analysis.png', bbox_inches='tight')
plt.show()

print("\nHigh-value targets (rate > 15%):")
print(job_stats[job_stats['rate'] > 15][['job', 'rate', 'total']].sort_values('rate', ascending=False).to_string(index=False))


#### Fig 3.3 – Subscription Rate by Balance Tier

In [ ]:
tier_order = ['NEGATIVE', 'LOW', 'MIDDLE', 'HIGH', 'PREMIUM']
tier_stats = (
    data.groupby('tier', observed=True)['y']
    .agg(total='count', subscribers=lambda x: (x == 'YES').sum())
    .assign(rate=lambda d: d['subscribers'] / d['total'] * 100)
    .reindex(tier_order)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = sns.color_palette('Blues_d', len(tier_stats))

axes[0].bar(tier_stats['tier'], tier_stats['total'], color=colors, edgecolor='white')
axes[0].set_title('Volume by Balance Tier')
axes[0].set_ylabel('Count')
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))

axes[1].bar(tier_stats['tier'], tier_stats['rate'], color=colors, edgecolor='white')
axes[1].set_title('Subscription Rate by Balance Tier')
axes[1].set_ylabel('Rate (%)')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
for ax in [axes[0], axes[1]]:
    ax.set_xlabel('Balance Tier')

plt.suptitle('Fig 3.3 – Balance Tier Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig3_3_balance_tier.png', bbox_inches='tight')
plt.show()

print("\nBalance Tier Summary:")
print(tier_stats[['tier', 'total', 'subscribers', 'rate']].to_string(index=False))


#### Fig 3.4 – Education Level Analysis

In [ ]:
edu_order = ['PRIMARY', 'SECONDARY', 'TERTIARY', 'UNKNOWN']
edu_stats = (
    data.groupby('education', observed=True)['y']
    .agg(total='count', subscribers=lambda x: (x == 'YES').sum())
    .assign(rate=lambda d: d['subscribers'] / d['total'] * 100)
    .reindex(edu_order)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(edu_stats))
bars_total = ax.bar(x - 0.2, edu_stats['total'] / 1000, 0.35,
                    label='Total (k)', color='#5B8DB8', alpha=0.85)
ax2 = ax.twinx()
bars_rate = ax2.plot(x, edu_stats['rate'], 'o--', color='#E67E22',
                     linewidth=2, markersize=8, label='Rate (%)')
ax.set_xticks(x)
ax.set_xticklabels(edu_stats['education'])
ax.set_ylabel("Contacts (thousands)")
ax2.set_ylabel("Subscription Rate (%)")
ax2.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title("Fig 3.4 – Education Level vs Subscription Rate", fontsize=13, fontweight='bold')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
plt.savefig('fig3_4_education.png', bbox_inches='tight')
plt.show()


### Q2 – When Should Campaigns Run?

#### Fig 4 – Monthly Campaign Volume vs Conversion Rate

In [ ]:
month_order = ['JAN','FEB','MAR','APR','MAY','JUN','JUL','AUG','SEP','OCT','NOV','DEC']

monthly = pd.crosstab(data['month'], data['y'])
monthly.columns = ['NO', 'YES']
monthly['total'] = monthly['YES'] + monthly['NO']
monthly['rate']  = monthly['YES'] / monthly['total'] * 100
monthly = monthly.reset_index()
monthly['month'] = pd.Categorical(monthly['month'], categories=month_order, ordered=True)
monthly = monthly.sort_values('month')

fig, ax1 = plt.subplots(figsize=(13, 5))
ax2 = ax1.twinx()

bars = ax1.bar(monthly['month'], monthly['total'],
               color=sns.color_palette('Blues_d', len(monthly)), edgecolor='white', alpha=0.8)
ax2.plot(monthly['month'], monthly['rate'], 'o-', color='#E74C3C', linewidth=2.5,
         markersize=8, label='Conversion Rate')
ax2.fill_between(range(len(monthly)), monthly['rate'], alpha=0.08, color='#E74C3C')

for bar, val in zip(bars, monthly['total']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
             f'{val:,}', ha='center', fontsize=7.5)

ax1.set_ylabel('Number of Contacts')
ax2.set_ylabel('Conversion Rate (%)')
ax2.yaxis.set_major_formatter(mtick.PercentFormatter())
ax1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax1.set_xlabel('Month')
ax1.set_title('Fig 4 – Campaign Volume vs Conversion Rate by Month', fontsize=13, fontweight='bold')
ax2.legend(loc='upper right')

# Annotate highest/lowest
best_m = monthly.loc[monthly['rate'].idxmax()]
ax2.annotate(f"Peak: {best_m['rate']:.1f}%",
             xy=(monthly['month'].tolist().index(best_m['month']), best_m['rate']),
             xytext=(monthly['month'].tolist().index(best_m['month']) + 0.5, best_m['rate'] + 1.5),
             arrowprops=dict(arrowstyle='->', color='black'), fontsize=9, color='darkred')

plt.tight_layout()
plt.savefig('fig4_monthly_campaigns.png', bbox_inches='tight')
plt.show()

print("\nMonthly Conversion Summary (sorted by rate):")
print(monthly[['month','total','YES','rate']].sort_values('rate', ascending=False).to_string(index=False))


### Q3 – Which Contact Mode Is Most Effective?

#### Fig 5 – Contact Type vs Subscription Rate

In [ ]:
contact_stats = (
    data.groupby('contact')['y']
    .agg(total='count', subs=lambda x: (x == 'YES').sum())
    .assign(rate=lambda d: d['subs'] / d['total'] * 100)
    .sort_values('rate', ascending=False)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ['#27AE60', '#E67E22', '#E74C3C']
bars1 = axes[0].bar(contact_stats['contact'], contact_stats['rate'], color=colors[:len(contact_stats)],
                    edgecolor='white')
axes[0].set_title('Subscription Rate by Contact Type')
axes[0].set_ylabel('Rate (%)')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
for bar, val in zip(bars1, contact_stats['rate']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')

# Stacked bar: volume breakdown
bottom_no = contact_stats['total'] - contact_stats['subs']
axes[1].bar(contact_stats['contact'], bottom_no, label='Did NOT Subscribe', color='#E74C3C', alpha=0.7)
axes[1].bar(contact_stats['contact'], contact_stats['subs'], bottom=bottom_no,
            label='Subscribed', color='#27AE60', alpha=0.9)
axes[1].set_title('Contact Volume & Subscription Mix')
axes[1].set_ylabel('Count')
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))
axes[1].legend()

plt.suptitle('Fig 5 – Contact Channel Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig5_contact_mode.png', bbox_inches='tight')
plt.show()

print("\nContact Summary:")
print(contact_stats[['contact','total','subs','rate']].to_string(index=False))
print("\n→ CELLULAR is the dominant and most effective channel.")
print("→ UNKNOWN contact likely indicates organic/word-of-mouth interest – higher intent.")


### Q4 – How Many Contacts Before Diminishing Returns?

In [ ]:
# Current campaign contacts
camp_stats = (
    data.groupby('campaign')['y']
    .apply(lambda x: (x == 'YES').mean() * 100)
    .reset_index(name='rate')
)
camp_capped = camp_stats[camp_stats['campaign'] <= 20].copy()

# Previous campaign contacts
prev_stats = (
    data.groupby('previous')['y']
    .apply(lambda x: (x == 'YES').mean() * 100)
    .reset_index(name='rate')
)
prev_capped = prev_stats[prev_stats['previous'] <= 15].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(camp_capped['campaign'], camp_capped['rate'], 'o-', color='#2980B9', linewidth=2.5, markersize=7)
axes[0].fill_between(camp_capped['campaign'], camp_capped['rate'], alpha=0.1, color='#2980B9')
axes[0].axvline(x=3, color='red', linestyle='--', linewidth=1.5, label='Recommended cap (3)')
axes[0].set_title('Subscription Rate vs Contacts This Campaign')
axes[0].set_xlabel('Number of Contacts (Current Campaign)')
axes[0].set_ylabel('Subscription Rate (%)')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[0].legend()

axes[1].plot(prev_capped['previous'], prev_capped['rate'], 'o-', color='#8E44AD', linewidth=2.5, markersize=7)
axes[1].fill_between(prev_capped['previous'], prev_capped['rate'], alpha=0.1, color='#8E44AD')
axes[1].axvline(x=5, color='red', linestyle='--', linewidth=1.5, label='Suggested review at 5')
axes[1].set_title('Subscription Rate vs Previous Campaign Contacts')
axes[1].set_xlabel('Previous Contacts')
axes[1].set_ylabel('Subscription Rate (%)')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].legend()

plt.suptitle('Fig 6 – Contact Frequency & Diminishing Returns', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig6_contact_frequency.png', bbox_inches='tight')
plt.show()

# Contact bucket analysis
bucket_stats = (
    data.groupby('campaign_bucket', observed=True)['y']
    .agg(total='count', subs=lambda x: (x == 'YES').sum())
    .assign(rate=lambda d: d['subs'] / d['total'] * 100)
    .reset_index()
)
print("\nContact Bucket Summary:")
print(bucket_stats.to_string(index=False))
print("\n→ Conversion rate drops sharply after 3–4 contacts in current campaign.")
print("→ Flag customers who have been called 4+ times for de-prioritisation.")


### Q5 – Does Previous Campaign Outcome Predict Future Success?

In [ ]:
pout_stats = (
    data.groupby('poutcome')['y']
    .agg(total='count', subs=lambda x: (x == 'YES').sum())
    .assign(rate=lambda d: d['subs'] / d['total'] * 100)
    .sort_values('rate', ascending=False)
    .reset_index()
)

# Previously-contacted vs not
prev_contact_stats = (
    data.groupby('previously_contacted')['y']
    .agg(total='count', subs=lambda x: (x == 'YES').sum())
    .assign(rate=lambda d: d['subs'] / d['total'] * 100)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors_p = [PALETTE['YES'] if o == 'SUCCESS' else '#F39C12' if o == 'OTHER' else '#E74C3C'
            for o in pout_stats['poutcome']]
bars = axes[0].bar(pout_stats['poutcome'], pout_stats['rate'], color=colors_p, edgecolor='white')
axes[0].set_title('Subscription Rate by Previous Outcome')
axes[0].set_ylabel('Rate (%)')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
for bar, val in zip(bars, pout_stats['rate']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')

colors_c = [PALETTE['YES'] if c == 'YES' else PALETTE['NO'] for c in prev_contact_stats['previously_contacted']]
bars2 = axes[1].bar(prev_contact_stats['previously_contacted'], prev_contact_stats['rate'],
                    color=colors_c, edgecolor='white')
axes[1].set_title('Subscription Rate: Previously Contacted?')
axes[1].set_ylabel('Rate (%)')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
for bar, val in zip(bars2, prev_contact_stats['rate']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')
axes[1].set_xlabel('Previously Contacted')

plt.suptitle('Fig 7 – Previous Campaign Signals', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig7_previous_outcome.png', bbox_inches='tight')
plt.show()

print("Previous Outcome Summary:")
print(pout_stats.to_string(index=False))
print("\nPreviously Contacted vs Not:")
print(prev_contact_stats.to_string(index=False))
print("\n→ Previous SUCCESS customers convert at 65%+ — re-engage them FIRST.")
print("→ Customers never previously contacted still have ~10% baseline rate.")


---
## VI. Customer Profiling

Build composite personas to guide campaign targeting.


In [ ]:
# ─── High-Propensity Profile ──────────────────────────────────────────────────
high = data[data['y'] == 'YES']
low  = data[data['y'] == 'NO']

print("=" * 60)
print("HIGH-PROPENSITY CUSTOMER PROFILE (Subscribers)")
print("=" * 60)
print(f"  Age (median)        : {high['age'].median():.0f} years")
print(f"  Balance (median)    : €{high['balance'].median():,.0f}")
print(f"  Top job             : {high['job'].mode()[0]}")
print(f"  Top age band        : {high['age_band'].mode()[0]}")
print(f"  Education           : {high['education'].mode()[0]}")
print(f"  Contact mode        : {high['contact'].mode()[0]}")
print(f"  Month contacted     : {high['month'].mode()[0]}")
print(f"  Campaign contacts   : {high['campaign'].median():.0f} (median)")
print(f"  Previously contacted: {(high['previously_contacted']=='YES').mean()*100:.1f}% were")
print(f"  Prior success       : {high['previous_success'].mean()*100:.1f}% had prior success")

print()
print("=" * 60)
print("LOW-PROPENSITY CUSTOMER PROFILE (Non-Subscribers)")
print("=" * 60)
print(f"  Age (median)        : {low['age'].median():.0f} years")
print(f"  Balance (median)    : €{low['balance'].median():,.0f}")
print(f"  Top job             : {low['job'].mode()[0]}")
print(f"  Top age band        : {low['age_band'].mode()[0]}")
print(f"  Education           : {low['education'].mode()[0]}")
print(f"  Contact mode        : {low['contact'].mode()[0]}")
print(f"  Month contacted     : {low['month'].mode()[0]}")
print(f"  Campaign contacts   : {low['campaign'].median():.0f} (median)")
print(f"  Previously contacted: {(low['previously_contacted']=='YES').mean()*100:.1f}% were")


#### Fig 8 – Customer Profile Heatmap (Subscription Rate by Job × Age Band)

In [ ]:
pivot = (
    data.groupby(['job', 'age_band'], observed=True)['y']
    .apply(lambda x: (x == 'YES').mean() * 100)
    .unstack('age_band')
)

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlGn',
            linewidths=0.5, linecolor='white', ax=ax, cbar_kws={'label': 'Subscription Rate (%)'})
ax.set_title('Fig 8 – Subscription Rate (%) by Job × Age Band', fontsize=13, fontweight='bold')
ax.set_xlabel('Age Band')
ax.set_ylabel('Job')
plt.tight_layout()
plt.savefig('fig8_profile_heatmap.png', bbox_inches='tight')
plt.show()
print("→ Darkest cells = highest-ROI targeting segments.")


#### Fig 9 – Segment ROI Waterfall: Contact Mode × Age Band

In [ ]:
seg = (
    data.groupby(['contact', 'age_band'], observed=True)['y']
    .apply(lambda x: (x == 'YES').mean() * 100)
    .reset_index(name='rate')
    .pivot(index='contact', columns='age_band', values='rate')
)

fig, ax = plt.subplots(figsize=(11, 5))
seg.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='white', width=0.7)
ax.set_title('Fig 9 – Subscription Rate by Contact Mode × Age Band', fontsize=13, fontweight='bold')
ax.set_ylabel('Subscription Rate (%)')
ax.set_xlabel('Contact Mode')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend(title='Age Band', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig('fig9_segment_roi.png', bbox_inches='tight')
plt.show()


---
## VII. Modelling Pipeline


### 7.1 Prepare Data

`duration` is dropped (target leakage).  
`age_band`, `tier`, `campaign_bucket`, `previously_contacted` are EDA helpers — the model
gets the raw numeric equivalents via the pipeline (which re-derives them as needed).


In [ ]:
DROP_COLS = ['duration', 'age_band', 'tier', 'campaign_bucket', 'previously_contacted',
             'previous_success']

datam = data.drop(columns=DROP_COLS, errors='ignore').copy()

Y = datam['y'].map({'NO': 0, 'YES': 1})
X = datam.drop('y', axis=1)

print(f"Features used: {list(X.columns)}")
print(f"Shape: {X.shape}  |  Class balance: {Y.mean():.3f} positive rate")


### 7.2 Feature Engineering Transformer (Sklearn-compatible)

In [ ]:
class BankFeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Derives new features in-pipeline so train/test leakage is impossible.
    
    New features created:
        contacted_before  – binary flag: was pdays != -1?
        month_num         – integer 1-12
        month_sin / cos   – cyclic encoding of month
    
    Dropped:
        month, pdays      – replaced by the above
    """

    MONTH_MAP = {'JAN':1,'FEB':2,'MAR':3,'APR':4,'MAY':5,'JUN':6,
                 'JUL':7,'AUG':8,'SEP':9,'OCT':10,'NOV':11,'DEC':12}

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()

        # ── pdays / contacted_before ───────────────────────────────────────────
        df['contacted_before'] = (df['pdays'] != -1).astype(int)
        df['pdays_clean']      = df['pdays'].replace(-1, 0)  # 0 = never contacted
        df = df.drop(columns=['pdays'])

        # ── Month cyclic encoding ──────────────────────────────────────────────
        df['month_num'] = df['month'].map(self.MONTH_MAP).fillna(6)
        df['month_sin'] = np.sin(2 * np.pi * df['month_num'] / 12)
        df['month_cos'] = np.cos(2 * np.pi * df['month_num'] / 12)
        df = df.drop(columns=['month'])

        return df

# Quick sanity-check
_fe = BankFeatureEngineer()
_X_test = _fe.fit_transform(X.head(3))
print("Feature-engineered columns:", list(_X_test.columns))


### 7.3 Preprocessing Pipelines

In [ ]:
# After feature engineering these are the column sets
NUM_FEATURES = ['age', 'balance', 'day', 'campaign', 'previous',
                'month_num', 'month_sin', 'month_cos', 'pdays_clean', 'contacted_before']

CAT_FEATURES = ['job', 'marital', 'education', 'default', 'housing',
                'loan', 'contact', 'poutcome']

# balance has extreme skew → PowerTransformer
BALANCE_FEATURE = ['balance']
OTHER_NUM       = [f for f in NUM_FEATURES if f != 'balance']

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  RobustScaler())
])

balance_pipeline = Pipeline([
    ('pt', PowerTransformer(method='yeo-johnson'))
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num',  num_pipeline,     OTHER_NUM),
    ('bal',  balance_pipeline, BALANCE_FEATURE),
    ('cat',  cat_pipeline,     CAT_FEATURES)
], remainder='drop')

print("✅  Preprocessor configured.")


### 7.4 Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

print(f"Train: {len(X_train):,} samples  |  Test: {len(X_test):,} samples")
print(f"Train positive rate: {y_train.mean():.3f}  |  Test positive rate: {y_test.mean():.3f}")


### 7.5 Model Comparison via Cross-Validation

In [ ]:
fe_step = ('fe', BankFeatureEngineer())
pp_step = ('pp', preprocessor)

models_to_compare = {
    'Logistic Regression': LogisticRegression(
        class_weight='balanced', max_iter=2000, random_state=42, C=0.1
    ),
    'Random Forest': RandomForestClassifier(
        class_weight='balanced', n_estimators=200, max_depth=8,
        random_state=42, n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', random_state=42, n_jobs=-1
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['roc_auc', 'f1', 'precision', 'recall', 'average_precision']

cv_results = {}
for name, model in models_to_compare.items():
    pipe = Pipeline([fe_step, pp_step, ('model', model)])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    cv_results[name] = {metric: scores[f'test_{metric}'].mean() for metric in scoring}
    print(f"[{name}]")
    for m in scoring:
        print(f"  {m:20s}: {cv_results[name][m]:.4f}")
    print()

cv_df = pd.DataFrame(cv_results).T.sort_values('roc_auc', ascending=False)
print("\n── CV Leaderboard ──────────────────────────────────────")
print(cv_df.round(4).to_string())


### 7.6 Fig 10 – Model Comparison Chart

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
metrics = ['roc_auc', 'f1', 'precision', 'recall', 'average_precision']
x = np.arange(len(metrics))
width = 0.25

for i, (model_name, scores) in enumerate(cv_results.items()):
    vals = [scores[m] for m in metrics]
    bars = ax.bar(x + i * width, vals, width, label=model_name, alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels(['ROC-AUC', 'F1', 'Precision', 'Recall', 'Avg Precision'])
ax.set_ylabel('Score')
ax.set_title('Fig 10 – 5-Fold CV Model Comparison', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1)
ax.legend()
ax.axhline(0.5, color='grey', linestyle='--', linewidth=0.8, label='Baseline')
plt.tight_layout()
plt.savefig('fig10_model_comparison.png', bbox_inches='tight')
plt.show()


### 7.7 Hyperparameter Tuning (Best Model)

In [ ]:
# ── Pick the best model from CV (XGBoost or RF typically wins) ──────────────
best_model_name = cv_df.index[0]
print(f"Best model from CV: {best_model_name}")

# XGBoost tuning grid
xgb_param_grid = {
    'model__n_estimators':    [200, 400],
    'model__max_depth':       [4, 6],
    'model__learning_rate':   [0.03, 0.07],
    'model__subsample':       [0.7, 0.9],
    'model__colsample_bytree':[0.7, 0.9],
    'model__min_child_weight':[1, 5],
}

rf_param_grid = {
    'model__n_estimators': [200, 400],
    'model__max_depth':    [6, 10, None],
    'model__min_samples_split': [2, 10],
    'model__min_samples_leaf':  [1, 5],
}

param_grid = xgb_param_grid if 'XGB' in best_model_name else rf_param_grid
base_model  = models_to_compare[best_model_name]

best_pipe = Pipeline([fe_step, pp_step, ('model', base_model)])

rscv = RandomizedSearchCV(
    estimator=best_pipe,
    param_distributions=param_grid,
    n_iter=16,
    cv=cv,
    scoring='roc_auc',
    refit=True,
    n_jobs=-1,
    random_state=42,
    verbose=1
)
rscv.fit(X_train, y_train)

print(f"\nBest params : {rscv.best_params_}")
print(f"Best ROC-AUC: {rscv.best_score_:.4f}")


### 7.8 Final Evaluation on Hold-Out Test Set

In [ ]:
y_pred       = rscv.predict(X_test)
y_pred_proba = rscv.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_pred_proba)
avg_prec = average_precision_score(y_test, y_pred_proba)
f1  = f1_score(y_test, y_pred)

print(f"Test ROC-AUC          : {roc_auc:.4f}")
print(f"Test Average Precision: {avg_prec:.4f}")
print(f"Test F1-Score         : {f1:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=['No Subscribe', 'Subscribe']))


#### Fig 11 – Confusion Matrix & ROC/PR Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['No Subscribe', 'Subscribe']).plot(
    ax=axes[0], colorbar=False, cmap='Blues'
)
axes[0].set_title('Confusion Matrix', fontsize=12, fontweight='bold')

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
axes[1].plot(fpr, tpr, lw=2, color='#2980B9', label=f'AUC = {roc_auc:.3f}')
axes[1].plot([0,1], [0,1], 'k--', lw=1)
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#2980B9')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontsize=12, fontweight='bold')
axes[1].legend()

# Precision-Recall curve
prec, rec, _ = precision_recall_curve(y_test, y_pred_proba)
axes[2].plot(rec, prec, lw=2, color='#27AE60', label=f'AP = {avg_prec:.3f}')
axes[2].axhline(y_test.mean(), color='red', linestyle='--', lw=1, label=f'Baseline ({y_test.mean():.3f})')
axes[2].fill_between(rec, prec, alpha=0.1, color='#27AE60')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title('Precision-Recall Curve', fontsize=12, fontweight='bold')
axes[2].legend()

plt.suptitle('Fig 11 – Model Evaluation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig11_evaluation.png', bbox_inches='tight')
plt.show()


#### Fig 12 – Feature Importance

In [ ]:
fi_result = permutation_importance(
    rscv, X_test, y_test,
    n_repeats=10, scoring='roc_auc', n_jobs=-1, random_state=42
)

# Recover feature names from the pipeline preprocessor
try:
    ohe_cats = (rscv.best_estimator_
                .named_steps['pp']
                .transformers_[2][1]
                .named_steps['ohe']
                .get_feature_names_out(CAT_FEATURES))
    feature_names = OTHER_NUM + BALANCE_FEATURE + list(ohe_cats)
except Exception:
    feature_names = [f'f{i}' for i in range(len(fi_result.importances_mean))]

fi_df = (pd.DataFrame({'feature': feature_names,
                        'importance': fi_result.importances_mean})
          .sort_values('importance', ascending=True)
          .tail(20))

fig, ax = plt.subplots(figsize=(10, 7))
colors = plt.cm.YlOrRd(np.linspace(0.3, 0.9, len(fi_df)))
ax.barh(fi_df['feature'], fi_df['importance'], color=colors)
ax.set_title('Fig 12 – Top 20 Feature Importances (Permutation, ROC-AUC)', fontsize=13, fontweight='bold')
ax.set_xlabel('Mean Importance Drop in ROC-AUC')
plt.tight_layout()
plt.savefig('fig12_feature_importance.png', bbox_inches='tight')
plt.show()


### 7.9 Propensity Score Output & Calibration Check

In [ ]:
# ── Calibration plot ──────────────────────────────────────────────────────────
prob_true, prob_pred = calibration_curve(y_test, y_pred_proba, n_bins=10, strategy='quantile')

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(prob_pred, prob_true, 'o-', color='#2980B9', lw=2, label='Model calibration')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect calibration')
ax.fill_between(prob_pred, prob_true, prob_pred,
                where=(prob_true > prob_pred), alpha=0.1, color='green', label='Underconfident zone')
ax.fill_between(prob_pred, prob_true, prob_pred,
                where=(prob_true < prob_pred), alpha=0.1, color='red', label='Overconfident zone')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Fig 13 – Calibration Curve', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('fig13_calibration.png', bbox_inches='tight')
plt.show()

# ── Propensity Score Distribution ─────────────────────────────────────────────
propensity_df = X_test.copy()
propensity_df['propensity_score'] = y_pred_proba
propensity_df['actual']           = y_test.values
propensity_df['predicted']        = y_pred

print("\nPropensity Score Summary:")
print(propensity_df['propensity_score'].describe().round(4))


#### Fig 14 – Propensity Score Distribution by Actual Label

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for label, grp in propensity_df.groupby('actual'):
    lname = 'Subscribed (y=1)' if label == 1 else 'Not Subscribed (y=0)'
    color = PALETTE['YES'] if label == 1 else PALETTE['NO']
    ax.hist(grp['propensity_score'], bins=40, alpha=0.6, label=lname, color=color, density=True)

ax.axvline(0.5, color='black', linestyle='--', lw=1.5, label='Default threshold (0.5)')
ax.set_xlabel('Predicted Propensity Score')
ax.set_ylabel('Density')
ax.set_title('Fig 14 – Propensity Score Distribution', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('fig14_propensity_distribution.png', bbox_inches='tight')
plt.show()


### 7.10 Threshold Optimisation for Business Use

In [ ]:
# Find threshold that maximises F1 (balances precision & recall for campaigns)
thresholds = np.linspace(0.05, 0.95, 91)
f1_scores  = [f1_score(y_test, (y_pred_proba >= t).astype(int)) for t in thresholds]
prec_scores = [precision_recall_curve(y_test, y_pred_proba)[0][
               np.argmax(precision_recall_curve(y_test, y_pred_proba)[1] >= t)] for t in thresholds]

best_idx       = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_f1        = f1_scores[best_idx]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(thresholds, f1_scores, color='#2980B9', lw=2, label='F1-Score')
ax.axvline(best_threshold, color='red', linestyle='--', lw=1.5,
           label=f'Optimal threshold = {best_threshold:.2f}')
ax.scatter([best_threshold], [best_f1], color='red', zorder=5, s=80)
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('F1-Score')
ax.set_title('Fig 15 – Threshold Optimisation', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('fig15_threshold_optimisation.png', bbox_inches='tight')
plt.show()

print(f"Optimal threshold for F1: {best_threshold:.2f}  |  F1 = {best_f1:.4f}")
print(f"\nWith this threshold:")
y_opt = (y_pred_proba >= best_threshold).astype(int)
print(classification_report(y_test, y_opt, target_names=['No Subscribe', 'Subscribe']))


---
## VIII. Campaign Targeting Summary
Use the propensity score to tier your entire customer base.


In [ ]:
# Score ALL records for illustration
all_scores = rscv.predict_proba(X)[:, 1]

scored = X.copy()
scored['propensity_score'] = all_scores
scored['actual_y']         = Y.values
scored['age_band']         = data['age_band'].values
scored['tier']             = data['tier'].values
scored['campaign_bucket']  = data['campaign_bucket'].values

scored['priority_segment'] = pd.cut(
    scored['propensity_score'],
    bins=[0, 0.15, 0.35, 0.60, 1.0],
    labels=['DO NOT CONTACT', 'LOW PRIORITY', 'MEDIUM PRIORITY', 'HIGH PRIORITY']
)

seg_summary = (
    scored.groupby('priority_segment', observed=True)
    .agg(
        count=('propensity_score', 'count'),
        avg_score=('propensity_score', 'mean'),
        actual_sub_rate=('actual_y', 'mean')
    )
    .assign(
        pct_of_base=lambda d: d['count'] / d['count'].sum() * 100
    )
    .reset_index()
)

print("\n─── Campaign Priority Segments ─────────────────────────────────────────────")
print(seg_summary.to_string(index=False))
print()
print("Campaign Recommendation:")
print("  🟢  HIGH PRIORITY     → Contact ASAP via Cellular")
print("  🟡  MEDIUM PRIORITY   → Contact via Cellular, limit to 2 attempts")
print("  🟠  LOW PRIORITY      → Include in mass digital/mail campaigns only")
print("  🔴  DO NOT CONTACT    → Suppress — cost outweighs expected return")


#### Fig 16 – Priority Segment Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

seg_colors = {'DO NOT CONTACT': '#E74C3C', 'LOW PRIORITY': '#F39C12',
              'MEDIUM PRIORITY': '#3498DB', 'HIGH PRIORITY': '#27AE60'}

colors_list = [seg_colors[s] for s in seg_summary['priority_segment']]

axes[0].bar(seg_summary['priority_segment'], seg_summary['count'],
            color=colors_list, edgecolor='white')
axes[0].set_title('Customer Count per Segment')
axes[0].set_ylabel('Count')
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))
axes[0].tick_params(axis='x', rotation=20)

axes[1].bar(seg_summary['priority_segment'],
            seg_summary['actual_sub_rate'] * 100,
            color=colors_list, edgecolor='white')
axes[1].set_title('Actual Subscription Rate per Segment')
axes[1].set_ylabel('Rate (%)')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].tick_params(axis='x', rotation=20)
for bar, val in zip(axes[1].patches, seg_summary['actual_sub_rate']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val*100:.1f}%', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Fig 16 – Campaign Priority Segments', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig16_priority_segments.png', bbox_inches='tight')
plt.show()


## IX. Save Model & Scored Output

In [ ]:
# ── Save best model ───────────────────────────────────────────────────────────
joblib.dump(rscv, 'propensity_model.pkl')
print("✅  Model saved → propensity_model.pkl")

# ── Save scored customer base ─────────────────────────────────────────────────
scored.to_csv('scored_customers.csv', index=False)
print("✅  Scored customers saved → scored_customers.csv")

# ── Load and test model ───────────────────────────────────────────────────────
loaded_model = joblib.load('propensity_model.pkl')
test_pred = loaded_model.predict_proba(X.head(5))[:, 1]
print(f"\n✅  Model reload test: {test_pred.round(3)}")


---
## X. Executive Summary & Recommendations

| Question | Finding | Recommendation |
|---|---|---|
| **Who to target?** | Students (28.7%), Retirees (22.8%), Tertiary-educated customers convert highest | Prioritise SENIOR + YOUNG-ADULT age bands in STUDENT/RETIRED job categories |
| **Best contact mode?** | Cellular (~15%) dominates; Unknown contact (~7%) shows latent interest | Use Cellular as primary channel; include UNKNOWN as warm leads |
| **How often to contact?** | Conversion drops sharply after 3 attempts in current campaign | Cap at 3 calls per campaign cycle; suppress 7+ contact records |
| **When to run campaigns?** | March & December have highest conversion rates despite low volume | Prioritise Q1 and Q4 budget allocation |
| **Previous campaign signal?** | Prior SUCCESS customers convert at 65%+ | Create a "re-engage priority" list from previous SUCCESS outcomes |
| **Never-contacted vs returning?** | Previously contacted customers convert ~3× higher | Use `previously_contacted = YES` as a strong positive signal |
| **Propensity model** | ROC-AUC > 0.80, F1 optimised via threshold tuning | Score all customers monthly; update model quarterly |

### Customer Archetypes

| Archetype | Profile | Action |
|---|---|---|
| 🟢 Gold Prospect | Retired/Student, Cellular contact, Prior SUCCESS, SENIOR age | Call within 48h, 1-2 attempts max |
| 🟡 Warm Prospect | Tertiary-educated, Management/Admin job, Previously contacted | Cellular, max 3 attempts |
| 🟠 Low Priority | Blue-collar, Middle-aged, HIGH campaign count | Digital only, no outbound calls |
| 🔴 Suppress | 7+ campaign contacts, LOW balance, FAILURE poutcome | Remove from campaign list |
